In [ ]:
import pandas as pd
import numpy as np
from datasist.structdata import detect_outliers
from sklearn.impute import SimpleImputer


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

df = pd.read_csv("train.csv", low_memory=False)
df


In [ ]:
df.info()

# Data Cleansing

1. **Not Useful Columns:**
   - `ID`, `Name`, `month` and `SSN` are not useful for the model. and Delete customers ID after being used

2. **Change Object Values to Float Values:**
   - `Age`, `Annual_Income`, `Num_of_Loan`, `Num_of_Delayed_Payment`, `Changed_Credit_Limit`, `Amount_invested_monthly`, `Outstanding_Debt`, `Credit_Mix`, `Monthly_Balance` are numerical but show as objects. 

3. **Values "__" in Columns:**
   - `Occupation` and `CreditMix` have values "__" that need to be addressed. And rewritten CreditMix category into float variables and Occupation into 15 different columns

4. **Outliers:**
   - Means and Delete Age rows that higher than 100 and lower than 0
   
5. **Num_Credit_Card Zeros:**
   - Num_Credit_Card that has 0

6. **Type_of_Loan Restructuring:**
   - `Type_of_Loan` needs to be rewritten as 8 columns.

7. **Negative Values in Num_Bank_Accounts:**
   - `Num_Bank_Accounts` Handle negative Values.

8. **Feature Engineering:**
   - `Credit_History_Age`, `Payment_of_Min_Amount`, `Payment_Behaviour`

9. **Missing Data:**
    - Need to fix missing Data

---

1. **Not Useful Columns:**

In [ ]:
del df['ID'] 
del df['Name']
del df['SSN']
del df['Month']

2. **Numeric Columns Incorrectly Typed as Categorical:**


In [ ]:
N_to_fix = ['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment'
            , 'Changed_Credit_Limit', 'Amount_invested_monthly'
            , 'Outstanding_Debt' , 'Monthly_Balance']

def fix_nums(num):
    try : 
        return float(num.replace("_",""))
    except :
        return np.nan
    
for col in N_to_fix :
    df[col] = df[col].apply(fix_nums)

In [ ]:
df.info()

5. **Num_Credit_Card Zeros:**

In [ ]:
df['Num_Credit_Card'].replace(0,1,inplace=True)

6. **Type_of_Loan Restructuring:**

In [ ]:
df[['Type_of_Loan']]


In [ ]:
for i in df['Type_of_Loan'].value_counts().head(9).index[1:] : 
    df[i] = df['Type_of_Loan'].str.contains(i)

del df['Type_of_Loan']

In [ ]:
df.info()

7. **Negative Values in Num_Bank_Accounts:**

In [ ]:
df['Num_Bank_Accounts'] = df['Num_Bank_Accounts'].apply(lambda x :abs (x))

8. **Feature Engineering:**
   - `Credit_History_Age`, `Payment_of_Min_Amount`, `Payment_Behaviour`.


**Credit_History_Age**

In [ ]:
print(df['Credit_History_Age'].sample(10))

In [ ]:
import re

def History_age(age):
    try: 
        numbers = re.findall(r'\d+', age)
        years = int(numbers[0]) if numbers else 0
        months = int(numbers[1]) if len(numbers) > 1 else 0
        return years * 12 + months
    except:
        return np.nan

df['Credit_History_Age'] = df['Credit_History_Age'].apply(History_age)

**Payment_of_Min_Amount**

In [ ]:
df['Payment_of_Min_Amount'].replace("NM","No",inplace=True)
df['Payment_of_Min_Amount'].value_counts()

**Payment_Behaviour**

In [ ]:
df['Payment_Behaviour']= df['Payment_Behaviour'].replace("!@9#%8",np.nan)
df['Payment_Behaviour'].value_counts()

3. **Values "__" in Columns:**

In [ ]:
df['Credit_Mix'].value_counts()

In [ ]:
m = {
    "Bad":0,
    "Standard":1,
    "Good":2,
    "_":np.nan
}
df['Credit_Mix'] = df['Credit_Mix'].map(m)
df['Credit_Mix'].value_counts()

In [ ]:
df['Occupation'].value_counts()

In [ ]:
customers_with_placeholder = df[df['Occupation'] == "_______"]['Customer_ID'].unique()
#mode for each customer group
mode_occupation_by_customer = df.groupby('Customer_ID')['Occupation'].apply(lambda x: x.mode()[0]).reset_index()
#replace '_______' with mode using merge
df = df.merge(mode_occupation_by_customer, on='Customer_ID', how='left', suffixes=('', '_mode'))
df['Occupation'] = df.apply(lambda row: row['Occupation_mode'] if row['Occupation'] == "_______" else row['Occupation'], axis=1)
df.drop(['Occupation_mode'], axis=1, inplace=True)


In [ ]:
df['Occupation'].value_counts()

In [ ]:
df['Occupation'] = df['Occupation'].replace("_______",df['Occupation'].mode()[0])
df['Occupation'].value_counts()


4. **Outliers:**

In [ ]:
df.dtypes

In [ ]:
Numericals = df.select_dtypes(exclude='object').columns[1:]

for col in Numericals:
    outliers_indices = detect_outliers(df, 0, [col])
    median = df[col].median()
    df.loc[outliers_indices, col] = median

In [ ]:
df = df[(df['Age'] >= 0) & (df['Age'] <= 100)]

9. **Advanced Handling Missing Data**

In [ ]:
for col in list(df.columns[-8:]):
    df[col] = df[col].astype(float)

In [ ]:
df['Customer_ID'] = pd.factorize(df['Customer_ID'])[0] + 1

In [ ]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=1)

In [ ]:
Numericals = df.select_dtypes(exclude='object').columns[1:]
Numericals

In [ ]:
imputer = SimpleImputer(strategy='median')
df[Numericals[1:]] = imputer.fit_transform(df[Numericals[1:]])

In [ ]:
imputer = SimpleImputer(strategy="most_frequent")
imputer.fit(df[['Payment_Behaviour']])
df[['Payment_Behaviour']] = imputer.transform(df[['Payment_Behaviour']])

In [ ]:
df.info()

In [ ]:
m = {
    "Poor":0,
    "Standard":1,
    "Good":2
}
df['Credit_Score'] = df['Credit_Score'].map(m)
del df['Customer_ID']

In [ ]:
df = pd.get_dummies(df,drop_first=False)

In [ ]:
# df.to_csv('cleaned_data.csv', index=False)

In [ ]:
print("Success")

In [ ]:
df.info()